In [1]:
import os
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                    .appName("SparkExample")
                    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        )
).getOrCreate()

hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f7f8e27a-c7fa-449e-b969-b20c73ace763;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5

In [2]:
# print("Активные Spark сессии:", spark.sparkContext.uiWebUrl)

jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD")
shops_df = (spark.read.format("jdbc")
		    .option("url", jdbc_url)
		    .option("user", db_user)
		    .option("password", db_password)
		    .option("dbtable", table_name)
		    .option("fetchsize", 1000)
		    .option("driver", "org.postgresql.Driver")
		    .load())


shops_df.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+



In [ ]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD")

shop_timezone_df = (spark.read.format("jdbc")
                            .option("url", jdbc_url)
                            .option("user", db_user)
                            .option("password", db_password)
                            .option("dbtable", table_name)
				            .option("fetchsize", 1000)
				            .option("driver", "org.postgresql.Driver")
				            .load())
                            
shop_timezone_df.show(truncate=False, vertical=False)                            


+-----+---------+
|plant|time_zone|
+-----+---------+
|842  |         |
|842  |RUS07    |
|843  |RUS04    |
|844  |         |
|845  |         |
|845  |RUS05    |
|847  |RUS03    |
|848  |RUS08    |
|848  |         |
|P847 |         |
|E103 |RUS08    |
|-134 |RUS04    |
|0    |         |
|0    |RUS08    |
|848  |         |
+-----+---------+



In [ ]:
query = """
WITH s1 AS (
    SELECT
        CAST(plant AS INT) AS shop_id,
        time_zone
    FROM shop_timezone
    WHERE CAST(plant AS INT) IS NOT NULL
),

shops_clean AS (
    SELECT
        CAST(st_id AS INT) AS shop_id,
        shop_name
    FROM shops
),

joined_dedup AS (
    SELECT
        s.shop_id,
        s.shop_name,
        t.time_zone,
        ROW_NUMBER() OVER (
            PARTITION BY s.shop_id
            ORDER BY s.shop_id
        ) AS rn
    FROM shops_clean s
    JOIN s1 t
        ON s.shop_id = t.shop_id
)

SELECT
    shop_id AS st_id,
    shop_name,
    CAST(
        CASE
            WHEN time_zone IS NULL OR time_zone = '' THEN 3
            ELSE SUBSTR(time_zone, 4)
        END
        AS INT
    ) AS tz_code
FROM joined_dedup
WHERE rn = 1
"""

spark.sql(query).show()


+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      3|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [21]:
from pyspark.sql import functions as F, Window

tz_clean = (
    shop_timezone_df
    .select(
        F.col('plant').cast("int").alias('shop_id'),
        F.col('time_zone')
    ).where(F.col('shop_id').isNotNull())
)

# tz_clean.show()

shops_clean = (
    shops_df
    .select(F.col("st_id").cast("int").alias("shop_id"), F.col("shop_name"))
)

w = Window.partitionBy("shop_id").orderBy("shop_id")

result = (
    shops_clean
    .join(tz_clean, on="shop_id", how="inner")
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") == 1)
    .select(
        F.col("shop_id").alias("st_id"),
        "shop_name",
        F.when((F.col("time_zone").isNull()) | (F.col("time_zone") == ""), F.lit(3))
         .otherwise(F.substring("time_zone", 4, 1000).cast("int"))
         .alias("tz_code")
    )
)

result.show()



+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      3|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [22]:
spark.stop()